# Customer Churn Analytics with AI

Colab-ready reproduction of the final project methodology: load → clean → EDA → leakage-safe train/validation/test modeling → validation-only operating threshold → untouched test evaluation.

In [ ]:
%pip install -q pandas==2.2.3 numpy==2.3.5 scikit-learn==1.8.0 scipy==1.17.0 matplotlib==3.10.8 seaborn==0.13.2 joblib==1.5.3

## 1. Load and verify the IBM Telco dataset

In [ ]:
import urllib.request
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
urllib.request.urlretrieve(DATA_URL, "telco_churn.csv")
df = pd.read_csv("telco_churn.csv")

assert df.shape == (7043, 21), df.shape
assert df["customerID"].is_unique
assert set(df["Churn"].unique()) == {"Yes", "No"}
print("raw data OK:", df.shape)
df.head()

## 2. Clean and engineer analysis features

In [ ]:
SERVICE_YES_COLS = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
missing_total = df["TotalCharges"].isna()
assert (df.loc[missing_total, "tenure"] == 0).all()
print("blank TotalCharges fixed:", int(missing_total.sum()))
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

df["churn_binary"] = (df["Churn"] == "Yes").astype(int)
df["tenure_band"] = pd.cut(
    df["tenure"], [-1, 6, 12, 24, 200], labels=["0-6", "7-12", "13-24", "25+"]
).astype(str)
df["num_services"] = ((df[SERVICE_YES_COLS] == "Yes").sum(axis=1) + df["InternetService"].ne("No").astype(int)).astype(int)
df["is_fiber"] = (df["InternetService"] == "Fiber optic").astype(int)
df["is_month_to_month"] = (df["Contract"] == "Month-to-month").astype(int)
df["is_electronic_check"] = (df["PaymentMethod"] == "Electronic check").astype(int)

assert len(df) == 7043
print("clean rows:", len(df), "| churn rate:", round(df["churn_binary"].mean(), 4))

## 3. Reproducible exploratory analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

contract = df.groupby("Contract")["churn_binary"].mean().reindex(["Month-to-month", "One year", "Two year"]) * 100
contract.plot(kind="bar", title="Churn rate by contract", ylabel="Churn rate (%)", rot=0)
plt.show()

tenure = df.groupby("tenure_band", observed=True)["churn_binary"].mean().reindex(["0-6", "7-12", "13-24", "25+"]) * 100
tenure.plot(kind="bar", title="Churn rate by tenure band", ylabel="Churn rate (%)", rot=0)
plt.show()

num = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "num_services",
       "is_fiber", "is_month_to_month", "is_electronic_check", "churn_binary"]
plt.figure(figsize=(9, 6))
sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlations")
plt.show()

## 4. Define the final model features

Exact duplicate indicators (`is_fiber`, `is_month_to_month`, `is_electronic_check`) are analysis-only. `TotalCharges` is also excluded from the live model because it cannot be reliably reconstructed from a short prediction form. One-hot encoding drops a reference category for cleaner coefficient interpretation.

In [ ]:
CAT = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod", "tenure_band"
]
NUM = ["SeniorCitizen", "tenure", "MonthlyCharges", "num_services"]
TARGET = "churn_binary"
X = df[CAT + NUM]
y = df[TARGET]
print("model input shape:", X.shape)

## 5. Split 60/20/20 and select the operating threshold on validation only

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=42
)

def make_model():
    return Pipeline([
        ("preprocess", ColumnTransformer([
            ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), CAT),
            ("num", StandardScaler(), NUM),
        ])),
        ("clf", LogisticRegression(max_iter=2000, random_state=42)),
    ])

selector = make_model().fit(X_train, y_train)
val_proba = selector.predict_proba(X_val)[:, 1]
rows = []
for threshold in np.round(np.arange(0.10, 0.61, 0.01), 2):
    pred = val_proba >= threshold
    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred),
        "f1": f1_score(y_val, pred),
    })
sweep = pd.DataFrame(rows)
eligible = sweep[sweep["recall"] >= 0.70]
chosen = eligible.sort_values(["precision", "f1", "threshold"], ascending=False).iloc[0]
OPERATING_THRESHOLD = float(chosen["threshold"])
print("chosen validation cutoff:", OPERATING_THRESHOLD)
chosen

## 6. Refit on train + validation and evaluate once on the untouched test set

In [ ]:
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay
)

model = make_model().fit(X_train_val, y_train_val)
test_proba = model.predict_proba(X_test)[:, 1]

def metrics_at(threshold):
    pred = test_proba >= threshold
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

default_metrics = metrics_at(0.50)
operating_metrics = metrics_at(OPERATING_THRESHOLD)
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))
print("Average precision:", round(average_precision_score(y_test, test_proba), 4))
print("Brier score:", round(brier_score_loss(y_test, test_proba), 4))
print("default:", default_metrics)
print("operating:", operating_metrics)

ConfusionMatrixDisplay.from_predictions(
    y_test, test_proba >= OPERATING_THRESHOLD, display_labels=["Retained", "Churned"], cmap="Blues"
)
plt.title(f"Confusion matrix @ {OPERATING_THRESHOLD:.2f}")
plt.show()

## 7. Save the fitted pipeline and summary artifacts

In [ ]:
import json
import joblib

joblib.dump(model, "churn_model.pkl")
summary = {
    "roc_auc": round(float(roc_auc_score(y_test, test_proba)), 4),
    "average_precision": round(float(average_precision_score(y_test, test_proba)), 4),
    "brier_score": round(float(brier_score_loss(y_test, test_proba)), 4),
    "operating_threshold": OPERATING_THRESHOLD,
    "test_default": default_metrics,
    "test_operating": operating_metrics,
}
with open("metrics.json", "w") as f:
    json.dump(summary, f, indent=2)
print("saved churn_model.pkl and metrics.json")

## 8. Interpretation

The model is useful for **risk ranking and retention prioritization**, not for proving causality. The production-style project repository adds a full Streamlit interface, coefficient-effect artifact, report, Docker deployment, CI, and validation script around this same methodology.